In [ ]:
import requests
import pandas as pd
import time
import os
from pathlib import Path
import boto3, json
import io

if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.config import load_config

CONFIG = load_config()

In [ ]:
"""
Deploiement (100% boto3, sans AWS CLI) du Lambda declencheur, attache au VPC
d'app-server, qui appelle l'API en IP privee (aucune exposition publique,
pas de Caddy, pas de secret transitant sur Internet).

Etapes :
  1. Recuperer subnet + VPC d'app-server
  2. Creer un security group dedie au Lambda
  3. Autoriser ce SG sur le port 8000 du SG d'app-server
  4. Role IAM d'execution (logs + acces ENI pour le VPC)
  5. Zipper et deployer la fonction (code = lambda_trigger_collect.py)
  6. Regle EventBridge Scheduler : tous les jours a 6h Europe/Paris
     (10 min apres le demarrage EC2 a 5h50 - regle "nappecast-start-instances" deja en place)

A adapter avant execution : la section CONFIG.
"""



# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
REGION = "eu-west-3"
ACCOUNT_ID = "844099234486"

PIPELINE_SECRET = os.environ["PIPELINE_SECRET"]  
FUNCTION_NAME = "nappecast-trigger-collect"
EXEC_ROLE_NAME = "nappecast-trigger-lambda-role"
LAMBDA_SG_NAME = "nappecast-lambda-trigger-sg"
SCHEDULE_NAME = "nappecast-daily-collect"

session = boto3.Session(
    aws_access_key_id=os.getenv("EC2_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("EC2_SECRET_ACCESS_KEY"),
    region_name=REGION,
)
ec2 = session.client("ec2")
iam = session.client("iam")
lambda_client = session.client("lambda")
scheduler = session.client("scheduler")

# ---------------------------------------------------------------------------
# 1. Subnet + VPC + SG d'app-server
# ---------------------------------------------------------------------------
resp = ec2.describe_instances(
    Filters=[{"Name": "tag:Name", "Values": ["app-server"]}, {"Name": "instance-state-name", "Values": ["running", "stopped"]}]
)
app_instance = resp["Reservations"][0]["Instances"][0]
app_vpc_id = app_instance["VpcId"]
app_subnet_id = app_instance["SubnetId"]
app_private_ip = app_instance["PrivateIpAddress"]
app_sg_id = app_instance["SecurityGroups"][0]["GroupId"]

print("app-server VPC:", app_vpc_id, "| subnet:", app_subnet_id, "| IP privee:", app_private_ip, "| SG:", app_sg_id)

# ---------------------------------------------------------------------------
# 2. Security group dedie au Lambda
# ---------------------------------------------------------------------------
try:
    sg = ec2.create_security_group(
        GroupName=LAMBDA_SG_NAME,
        Description="SG du Lambda declencheur du pipeline NappeCast",
        VpcId=app_vpc_id,
    )
    lambda_sg_id = sg["GroupId"]
    print("SG Lambda cree :", lambda_sg_id)
except ec2.exceptions.ClientError as e:
    if "InvalidGroup.Duplicate" in str(e):
        existing = ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [LAMBDA_SG_NAME]}, {"Name": "vpc-id", "Values": [app_vpc_id]}]
        )
        lambda_sg_id = existing["SecurityGroups"][0]["GroupId"]
        print("SG Lambda deja existant :", lambda_sg_id)
    else:
        raise

# ---------------------------------------------------------------------------
# 3. Autoriser ce SG sur le port 8000 du SG d'app-server
# ---------------------------------------------------------------------------
try:
    ec2.authorize_security_group_ingress(
        GroupId=app_sg_id,
        IpPermissions=[
            {
                "IpProtocol": "tcp",
                "FromPort": 8000,
                "ToPort": 8000,
                "UserIdGroupPairs": [{"GroupId": lambda_sg_id, "Description": "Lambda trigger pipeline"}],
            }
        ],
    )
    print("Regle ajoutee sur le SG app-server pour le port 8000")
except ec2.exceptions.ClientError as e:
    if "InvalidPermission.Duplicate" in str(e):
        print("Regle deja presente")
    else:
        raise

# ---------------------------------------------------------------------------
# 4. Role IAM d'execution (logs + ENI pour VPC)
# ---------------------------------------------------------------------------
lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}

try:
    exec_role = iam.create_role(
        RoleName=EXEC_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description="Role d'execution du Lambda declencheur NappeCast (VPC)",
    )
    exec_role_arn = exec_role["Role"]["Arn"]
    print("Role d'execution cree :", exec_role_arn)
except iam.exceptions.EntityAlreadyExistsException:
    exec_role_arn = iam.get_role(RoleName=EXEC_ROLE_NAME)["Role"]["Arn"]
    print("Role d'execution deja existant :", exec_role_arn)

iam.attach_role_policy(
    RoleName=EXEC_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
# Necessaire pour qu'un Lambda attache a un VPC puisse creer/gerer ses ENI
iam.attach_role_policy(
    RoleName=EXEC_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaVPCAccessExecutionRole",
)

time.sleep(10)

In [ ]:
import json

iam = session.client("iam")  # meme session que celle utilisee pour EC2/Scheduler

lambda_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "lambda:CreateFunction",
                "lambda:UpdateFunctionCode",
                "lambda:UpdateFunctionConfiguration",
                "lambda:GetFunction",
                "lambda:InvokeFunction",
                "lambda:DeleteFunction",
                "lambda:TagResource",
            ],
            "Resource": "arn:aws:lambda:eu-west-3:844099234486:function:nappecast-*",
        },
        {
            "Effect": "Allow",
            "Action": "iam:PassRole",
            "Resource": [
                "arn:aws:iam::844099234486:role/nappecast-trigger-lambda-role",
                "arn:aws:iam::844099234486:role/nappecast-scheduler-invoke-trigger-role",
            ],
        },
    ],
}

iam.put_user_policy(
    UserName="nappecast-admin",
    PolicyName="nappecast-lambda-deploy",
    PolicyDocument=json.dumps(lambda_policy),
)
print("Policy Lambda attachee a nappecast-admin")

In [ ]:
# ---------------------------------------------------------------------------
# 5. Creation / mise a jour de la fonction Lambda (attachee au VPC) + variables environnnement
# ---------------------------------------------------------------------------
LAMBDA_CODE_FILE = "/home/ronanguilloueee/NappeCast/src/lambda/lambda_trigger_collect.py"


with open(LAMBDA_CODE_FILE, "rb") as f:
    code_content = f.read()

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", code_content)
zip_bytes = zip_buffer.getvalue()

env_vars = {
    "APP_PRIVATE_IP": app_private_ip,
    "PIPELINE_SECRET": PIPELINE_SECRET,
    "PIPELINE_PATH": "/pipeline/collect",
    "PORT": "8000",
}

try:
    fn = lambda_client.create_function(
        FunctionName=FUNCTION_NAME,
        Runtime="python3.12",
        Role=exec_role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=180,
        MemorySize=256,
        Environment={"Variables": env_vars},
        VpcConfig={"SubnetIds": [app_subnet_id], "SecurityGroupIds": [lambda_sg_id]},
        Description="Declenche POST /pipeline/collect sur app-server (IP privee, interne au VPC)",
    )
    function_arn = fn["FunctionArn"]
    print("Fonction Lambda creee :", function_arn)
except lambda_client.exceptions.ResourceConflictException:
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_bytes)
    lambda_client.update_function_configuration(
        FunctionName=FUNCTION_NAME,
        Environment={"Variables": env_vars},
        VpcConfig={"SubnetIds": [app_subnet_id], "SecurityGroupIds": [lambda_sg_id]},
    )
    function_arn = lambda_client.get_function(FunctionName=FUNCTION_NAME)["Configuration"]["FunctionArn"]
    print("Fonction Lambda existante, mise a jour :", function_arn)

# ---------------------------------------------------------------------------
# 6. Role IAM pour qu'EventBridge Scheduler invoque ce Lambda
# ---------------------------------------------------------------------------
SCHEDULER_ROLE_NAME = "nappecast-scheduler-invoke-trigger-role"
scheduler_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "scheduler.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
try:
    scheduler_role = iam.create_role(
        RoleName=SCHEDULER_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(scheduler_trust_policy),
    )
    scheduler_role_arn = scheduler_role["Role"]["Arn"]
    print("Role scheduler cree :", scheduler_role_arn)
except iam.exceptions.EntityAlreadyExistsException:
    scheduler_role_arn = iam.get_role(RoleName=SCHEDULER_ROLE_NAME)["Role"]["Arn"]
    print("Role scheduler deja existant :", scheduler_role_arn)

iam.put_role_policy(
    RoleName=SCHEDULER_ROLE_NAME,
    PolicyName="nappecast-invoke-trigger-lambda",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": function_arn}],
        }
    ),
)

time.sleep(10)

# ---------------------------------------------------------------------------
# 7. Regle EventBridge Scheduler : tous les jours a 6h Europe/Paris
#    (apres la regle "nappecast-start-instances" a 5h50)
# ---------------------------------------------------------------------------
schedule_params = dict(
    Name=SCHEDULE_NAME,
    ScheduleExpression="cron(0 6 * * ? *)",
    ScheduleExpressionTimezone="Europe/Paris",
    FlexibleTimeWindow={"Mode": "OFF"},
    Target={"Arn": function_arn, "RoleArn": scheduler_role_arn, "Input": json.dumps({})},
    Description="Declenche le pipeline de collecte via Lambda (appel interne VPC) chaque matin a 6h Paris",
)
try:
    scheduler.create_schedule(**schedule_params)
    print("Schedule cree :", SCHEDULE_NAME)
except scheduler.exceptions.ConflictException:
    scheduler.update_schedule(**schedule_params)
    print("Schedule mis a jour :", SCHEDULE_NAME)

In [ ]:
# ---------------------------------------------------------------------------
# 8. Test manuel immediat (optionnel - app-server doit deja tourner)
# ---------------------------------------------------------------------------
test = input("Invoquer le Lambda maintenant pour tester ? (o/n) ")
if test.lower() == "o":
    invoke_resp = lambda_client.invoke(
        FunctionName=FUNCTION_NAME,
        InvocationType="RequestResponse",
        Payload=json.dumps({}).encode("utf-8"),
    )
    print(json.loads(invoke_resp["Payload"].read()))